# Singlish Hate Speech — Guardrails AI Demo

This notebook demonstrates how to package a hate speech classifier as a **Guardrails AI custom validator** — making it composable, deployable, and swappable into any LLM pipeline.

## Motivation

Standard LLM safety tools (LlamaGuard, Perspective API, dehatebert) are trained on English/Western datasets and **miss Singapore-specific hate speech**:
- Singlish slurs: `ah tiong`, `bangla`, `ang moh` used offensively
- Code-switching: racial insults embedded in casual Singlish sentences
- Cultural context: what looks like casual Singlish may be targeted hate

The **Guardrails approach** wraps your fine-tuned SG-specific model as a modular validator that any developer can plug into their LLM app — no ML expertise needed.

## Architecture

```
User text / LLM output
        │
        ▼
┌─────────────────────────────────────┐
│  Guard (input or output)            │
│  ├── SinglishHateSpeech validator   │  ← your fine-tuned model
│  ├── ToxicLanguage validator        │  ← from Guardrails Hub
│  └── ProfanityFree validator        │  ← from Guardrails Hub
└─────────────────────────────────────┘
        │
        ▼
  PassResult → safe content delivered
  FailResult → fix / filter / exception
```

## Notebook Structure

| Section | What it covers |
|---------|----------------|
| 1. Setup | Install Guardrails, load models |
| 2. Dataset preview | Explore the SG synthetic dataset |
| 3. Baseline gap | Show where off-the-shelf guards fail on Singlish |
| 4. Custom validator | Build `SinglishHateSpeech` wrapping HateXplain RoBERTa |
| 5. Guard composition | Combine SinglishHateSpeech + ToxicLanguage |
| 6. Input guard demo | Block hate speech before it reaches an LLM |
| 7. Output guard demo | Filter LLM outputs containing Singlish hate speech |
| 8. Benchmark | Evaluate guard accuracy on the SG test set |
| 9. Swap to fine-tuned | Slot in your fine-tuned model with zero code change |
| 10. Hub submission guide | How to publish your validator to the Guardrails Hub |

> **Note on models:** This notebook uses `shareeep/roberta-base-fine-tuned-hatexplain` as a placeholder.
> When fine-tuning (Phase 1 of the project plan) is complete, swap to your SG-specific model in Section 9.

---
## Section 1: Setup

```bash
# Create a new environment
uv venv .venv-guardrails && source .venv-guardrails/bin/activate
uv add guardrails-ai transformers torch scikit-learn pandas tqdm ipykernel
python -m ipykernel install --user --name=guardrails --display-name="Guardrails"

# Configure Guardrails (creates ~/.guardrailsrc)
guardrails configure --token YOUR_GUARDRAILS_TOKEN  # or --no-metrics for offline

# Install hub validators we'll use
guardrails hub install hub://guardrails/toxic_language --quiet
```

In [ ]:
# Verify installations
import importlib, sys

required = ['guardrails', 'transformers', 'torch', 'sklearn', 'pandas']
for pkg in required:
    try:
        importlib.import_module(pkg)
        print(f'  {pkg}: OK')
    except ImportError:
        print(f'  {pkg}: MISSING — run pip install {pkg}')

import guardrails
print(f'\nGuardrails version: {guardrails.__version__}')

In [ ]:
# Core imports
import json
import os
import torch
import pandas as pd
from collections import Counter
from typing import Any, Dict, Optional
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from guardrails import Guard
from guardrails.validator_base import (
    FailResult,
    PassResult,
    ValidationResult,
    Validator,
    register_validator,
)

LABEL_NAMES = ['hate', 'offensive', 'normal']
print('Imports OK.')

---
## Section 2: Dataset Preview

The SG synthetic dataset covers 20 Singlish hate speech terms across three target groups.

In [ ]:
DATA_BASE = 'data/singlish/synthetic/20_terms/split_60_20_20'

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

sg_train = load_jsonl(f'{DATA_BASE}/train.jsonl')
sg_val   = load_jsonl(f'{DATA_BASE}/val.jsonl')
sg_test  = load_jsonl(f'{DATA_BASE}/test.jsonl')

df_test = pd.DataFrame(sg_test)

print(f'Train: {len(sg_train)}  Val: {len(sg_val)}  Test: {len(sg_test)}')
print(f'\nLabel distribution (test):')
print(df_test['label'].value_counts().to_string())
print(f'\nTarget groups (test):')
print(df_test['target_group'].value_counts().to_string())
print(f'\nUnique Singlish terms: {df_test["source_term"].nunique()}')
print(f'Terms: {", ".join(sorted(df_test["source_term"].unique()))}')

In [ ]:
# Show one example per label
print('Example rows from test set:')
print('='*80)
for label in LABEL_NAMES:
    example = df_test[df_test['label'] == label].iloc[0]
    print(f'\n[{label.upper()}]')
    print(f'  Text:   {example["text"]}')
    print(f'  Term:   {example["source_term"]}')
    print(f'  Target: {example["target_group"]}')

---
## Section 3: The Baseline Gap

Demonstrate where off-the-shelf guards fail on Singlish. We test `ToxicLanguage` (toxic-bert) on examples from the SG dataset.

> This is the motivation for building a Singlish-specific validator.

In [ ]:
# Test with ToxicLanguage from Guardrails Hub
# Install first: guardrails hub install hub://guardrails/toxic_language
try:
    from guardrails.hub import ToxicLanguage

    toxic_guard = Guard().use(
        ToxicLanguage,
        threshold=0.5,
        validation_method='sentence',
        on_fail='noop',  # Don't raise, just log
    )

    # Hand-picked hard cases from the SG dataset
    gap_examples = [
        # These are labelled 'hate' — toxic-bert should flag them
        ('hate', 'those ah tiong always come here take our jobs lor'),
        ('hate', 'all the bangla workers dirty and smelly one, should ban them'),
        ('hate', 'malays always so lazy, that\'s why poor one'),
        # These are 'offensive' (Singlish expletives) — should NOT be flagged as hate
        ('offensive', 'nabeh, you damn stupid lah'),
        ('offensive', 'wah lan eh, you spoil my mood again'),
        # These are 'normal' — should not be flagged at all
        ('normal', 'wah lao eh the mee pok today damn shiok sia'),
        ('normal', 'eh bro later we go makan together can?'),
    ]

    print('ToxicLanguage (toxic-bert) on Singlish examples:')
    print(f'{"True":<12} {"Flagged?":<10} Text')
    print('-'*80)

    for true_label, text in gap_examples:
        result = toxic_guard.validate(text)
        flagged = not result.validation_passed
        correct = (true_label == 'hate') == flagged
        marker = '' if correct else '  <- MISS'
        print(f'{true_label:<12} {str(flagged):<10} {text[:60]}{marker}')

except ImportError:
    print('ToxicLanguage not installed.')
    print('Run: guardrails hub install hub://guardrails/toxic_language')
    print('\nSkipping gap demo — proceeding to custom validator.')

In [ ]:
# Quantify the gap on the full test set
try:
    from sklearn.metrics import classification_report

    y_true, y_pred_toxic = [], []

    for row in tqdm(sg_test[:50], desc='ToxicLanguage on SG test'):
        result = toxic_guard.validate(row['text'])
        y_true.append(row['label'])
        # ToxicLanguage is binary — map to our 3-class: flagged=hate, not flagged=normal
        y_pred_toxic.append('hate' if not result.validation_passed else 'normal')

    print('ToxicLanguage on SG test set (first 50 rows, binary approximation):')
    print(classification_report(y_true, y_pred_toxic, labels=['hate', 'offensive', 'normal'], zero_division=0))
    print('NOTE: ToxicLanguage cannot distinguish offensive from normal — it has no "offensive" class.')

except NameError:
    print('Skipped — ToxicLanguage not loaded.')

---
## Section 4: Custom `SinglishHateSpeech` Validator

We implement a Guardrails `Validator` subclass that wraps any HuggingFace hate speech model.

**Placeholder model:** `shareeep/roberta-base-fine-tuned-hatexplain`  
**Final model:** Your SG-fine-tuned SingBERT or RoBERTa (swap in Section 9)

In [ ]:
# ── Model loading helper (shared by all guard instances) ────────────────

_model_cache: Dict[str, tuple] = {}

def load_classifier(model_name: str):
    """Load and cache a HuggingFace sequence classification model."""
    if model_name not in _model_cache:
        print(f'Loading {model_name}...')
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(model_name)
        model.eval()
        _model_cache[model_name] = (tokenizer, model)
        print(f'  Labels: {model.config.id2label}')
    return _model_cache[model_name]


print('Model cache helper ready.')

In [ ]:
@register_validator(name='singlish-hate-speech', data_type='string')
class SinglishHateSpeech(Validator):
    """
    Detects Singlish hate speech using a HuggingFace sequence classification model.

    The validator flags text as hate speech when the model's confidence for the
    'hate' class exceeds `threshold`. It is Singapore-context aware: Singlish
    expletives (nabeh, cb, wah lan) are offensive but NOT hate speech, and this
    distinction is captured by a 3-class model trained on SG data.

    Args:
        model_name (str): HuggingFace model ID. Defaults to HateXplain RoBERTa
            as a placeholder until the SG fine-tuned model is published.
        hate_threshold (float): Confidence threshold for the 'hate' label [0, 1].
            Higher = stricter (fewer false positives, more false negatives).
        hate_label (str): The label name the model uses for hate speech.
            HateXplain RoBERTa uses 'hatespeech'; SG fine-tuned model uses 'hate'.
        on_fail: Guardrails on_fail action. Options:
            - 'exception' — raise ValidationError
            - 'fix'       — replace with fix_value (redacted message)
            - 'filter'    — remove from structured output
            - 'refrain'   — return None
            - 'noop'      — pass through unchanged, log the failure
    """

    # Placeholder model — swap for your SG fine-tuned model (see Section 9)
    DEFAULT_MODEL = 'shareeep/roberta-base-fine-tuned-hatexplain'

    def __init__(
        self,
        model_name: str = DEFAULT_MODEL,
        hate_threshold: float = 0.6,
        hate_label: str = 'hatespeech',  # HateXplain label; change to 'hate' for SG model
        on_fail: Optional[str] = None,
    ):
        super().__init__(on_fail=on_fail)
        self.model_name = model_name
        self.hate_threshold = hate_threshold
        self.hate_label = hate_label

        # Load model at init time — cached across instances with same model_name
        self._tokenizer, self._model = load_classifier(model_name)

    def _get_hate_prob(self, text: str) -> float:
        """Run inference and return the probability for the hate label."""
        inputs = self._tokenizer(
            text, return_tensors='pt', truncation=True, max_length=128
        )
        with torch.no_grad():
            logits = self._model(**inputs).logits
        probs = torch.softmax(logits, dim=1)[0]

        # Map label names to indices from model config
        label_to_id = {v: k for k, v in self._model.config.id2label.items()}
        hate_idx = label_to_id.get(self.hate_label, 0)
        return probs[hate_idx].item()

    def validate(self, value: Any, metadata: Dict = {}) -> ValidationResult:
        """
        Validate a string for Singlish hate speech.

        Returns PassResult if the text is safe, FailResult otherwise.
        The FailResult includes:
          - error_message: confidence score and detected label
          - fix_value: a redacted replacement string
        """
        if not isinstance(value, str) or not value.strip():
            return PassResult()

        hate_prob = self._get_hate_prob(value)

        if hate_prob >= self.hate_threshold:
            return FailResult(
                error_message=(
                    f'Singlish hate speech detected '
                    f'(model={self.model_name.split("/")[-1]}, '
                    f'hate_prob={hate_prob:.3f}, '
                    f'threshold={self.hate_threshold})'
                ),
                fix_value='[Content removed: hate speech detected]',
            )

        return PassResult()


print('SinglishHateSpeech validator defined.')
print(f'Registered as: singlish-hate-speech')

In [ ]:
# ── Smoke test the validator directly ──────────────────────────────────
validator = SinglishHateSpeech(hate_threshold=0.6, on_fail='noop')

smoke_tests = [
    ('hate',      'those ah tiong always come here take our jobs'),
    ('hate',      'all the bangla workers dirty and smelly one'),
    ('offensive', 'nabeh, you damn stupid lah'),
    ('offensive', 'cb, you sabo me again'),
    ('normal',    'wah lao eh, mee pok today damn shiok sia'),
    ('normal',    'eh bro later we go makan together can?'),
]

print('Direct validator smoke test (placeholder model — accuracy will improve with SG fine-tune):')
print(f'{"True":<12} {"Result":<8} {"Hate prob":<12} Text')
print('-'*80)

for true_label, text in smoke_tests:
    prob = validator._get_hate_prob(text)
    result = validator.validate(text)
    passed = isinstance(result, PassResult)
    correct = (true_label == 'hate') == (not passed)
    marker = '' if correct else '  <- MISS'
    print(f'{true_label:<12} {"PASS" if passed else "FAIL":<8} {prob:<12.3f} {text[:55]}{marker}')

---
## Section 5: Guard Composition

Combine `SinglishHateSpeech` with other Hub validators into a single composable Guard.
Each validator runs independently — a text can fail any one of them.

In [ ]:
# ── Guard 1: SinglishHateSpeech only (core validator) ─────────────────
sg_guard = Guard(name='singlish-hate-speech-guard').use(
    SinglishHateSpeech,
    hate_threshold=0.6,
    on_fail='fix',  # Replace with redacted message
)

print('Guard 1 (SinglishHateSpeech only):')
result = sg_guard.validate('malays always so lazy that is why poor one')
print(f'  Input:  malays always so lazy that is why poor one')
print(f'  Passed: {result.validation_passed}')
print(f'  Output: {result.validated_output}')
print(f'  Errors: {[e.msg for e in (result.error or [])]}')

In [ ]:
# ── Guard 2: Composite — SinglishHateSpeech + ToxicLanguage ───────────
# Catches both: SG-specific hate AND generic English toxicity
try:
    from guardrails.hub import ToxicLanguage

    composite_guard = Guard(name='sg-composite-guard').use_many(
        SinglishHateSpeech(hate_threshold=0.6, on_fail='fix'),
        ToxicLanguage(threshold=0.5, validation_method='sentence', on_fail='fix'),
    )

    test_inputs = [
        'those ah tiong come here take our jobs',     # SG hate
        'you are a fucking idiot',                    # Generic toxicity (English)
        'wah lao eh damn shiok leh',                  # Normal Singlish
        'nabeh you stupid lah',                       # Offensive but not hate
    ]

    print('Composite guard (SinglishHateSpeech + ToxicLanguage):')
    print(f'{"Input":<50} {"Passed":<8} Output')
    print('-'*100)

    for text in test_inputs:
        result = composite_guard.validate(text)
        out = result.validated_output or '[filtered]'
        print(f'{text:<50} {str(result.validation_passed):<8} {out}')

except ImportError:
    print('ToxicLanguage not installed — showing SinglishHateSpeech-only guard.')
    print('Run: guardrails hub install hub://guardrails/toxic_language')

---
## Section 6: Input Guard Demo

Use the guard to screen **user inputs** before they reach an LLM.
This prevents hate speech from being processed or amplified by the model.

In [ ]:
def simulate_input_guard(user_message: str, guard: Guard) -> dict:
    """
    Simulate an input guard in a chatbot pipeline.
    In production this would sit before your LLM call.
    """
    result = guard.validate(user_message)

    if result.validation_passed:
        # Safe — forward to LLM (simulated here)
        llm_response = f'[LLM processed]: Response to "{user_message[:40]}..."'
        return {'status': 'allowed', 'llm_response': llm_response}
    else:
        # Blocked — return rejection message without touching LLM
        return {
            'status': 'blocked',
            'reason': 'Hate speech detected in input',
            'llm_response': 'Sorry, I cannot process that request.',
        }


input_guard = Guard(name='input-guard').use(
    SinglishHateSpeech, hate_threshold=0.6, on_fail='refrain'
)

user_inputs = [
    'can you help me write an email to my team?',
    'those bangla workers should be deported back',
    'write me a birthday message lah, something shiok',
    'why do indians always smell like curry sia',
]

print('Input Guard Simulation:')
print('='*70)
for msg in user_inputs:
    outcome = simulate_input_guard(msg, input_guard)
    status_icon = 'ALLOWED' if outcome['status'] == 'allowed' else 'BLOCKED'
    print(f'\n[{status_icon}] "{msg}"')
    if outcome['status'] == 'blocked':
        print(f'  Reason: {outcome["reason"]}')
    print(f'  Response: {outcome["llm_response"][:80]}')

---
## Section 7: Output Guard Demo

Use the guard to screen **LLM outputs** before they reach the user.
Useful when the LLM may hallucinate or generate harmful content from ambiguous prompts.

In [ ]:
output_guard = Guard(name='output-guard').use(
    SinglishHateSpeech, hate_threshold=0.6, on_fail='fix'
)

# Simulate LLM outputs that may contain hate speech
simulated_llm_outputs = [
    # Safe outputs
    'Singapore is a beautiful multicultural city. People from all backgrounds live here.',
    'The hawker centre has amazing food lah. Must try the char kway teow!',
    # Potentially unsafe outputs
    'Some people believe that certain ethnic groups are lazy and unreliable.',
    'The neighbourhood has many bangla workers who are often seen as dirty and disruptive.',
]

print('Output Guard Simulation:')
print('='*70)
for output in simulated_llm_outputs:
    result = output_guard.validate(output)
    print(f'\nOriginal:  {output[:80]}')
    print(f'Passed:    {result.validation_passed}')
    print(f'Delivered: {result.validated_output[:80]}')

---
## Section 8: Benchmark — Guard Accuracy on SG Test Set

Evaluate the guard's classification accuracy on the 108-row SG synthetic test set.
This is the same test set used to benchmark fine-tuned models, enabling direct comparison.

In [ ]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix

# Guard in noop mode — we want to see predictions without modifying output
eval_validator = SinglishHateSpeech(
    hate_threshold=0.6,
    on_fail='noop',
)
eval_guard = Guard(name='eval-guard').use(eval_validator)

y_true, y_pred_guard = [], []

for row in tqdm(sg_test, desc='Guard eval on SG test set'):
    hate_prob = eval_validator._get_hate_prob(row['text'])
    result = eval_guard.validate(row['text'])

    # Map binary guard output to 3-class:
    # FAIL (hate prob high) → 'hate'
    # PASS → 'normal' (guard doesn't distinguish offensive from normal)
    pred = 'hate' if not result.validation_passed else 'normal'

    y_true.append(row['label'])
    y_pred_guard.append(pred)

print(f'\n{"="*60}')
print(f'  Placeholder Guard (HateXplain RoBERTa) on SG Test Set')
print(f'  Note: Binary guard — cannot classify "offensive" separately')
print('='*60)
print(classification_report(y_true, y_pred_guard, labels=['hate', 'offensive', 'normal'], zero_division=0))

macro_f1 = f1_score(y_true, y_pred_guard, labels=['hate', 'offensive', 'normal'], average='macro', zero_division=0)
print(f'Macro F1: {macro_f1:.4f}')
print(f'\n(Expected: low offensive F1 — binary guard cannot detect this class)')
print(f'(After SG fine-tune swap in Section 9, this will become 3-class)')

In [ ]:
# Threshold sensitivity analysis — find the optimal hate_threshold
from sklearn.metrics import f1_score as skf1

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

# Pre-compute hate probabilities for all test rows
hate_probs = [eval_validator._get_hate_prob(row['text']) for row in tqdm(sg_test, desc='Pre-computing probs')]

print('\nThreshold sensitivity (hate F1 on SG test set):')
print(f'{"Threshold":<12} {"Hate F1":<12} {"Normal F1":<12} {"Macro F1":<12} {"Precision":<12} {"Recall"}')
print('-'*75)

for thresh in thresholds:
    preds = ['hate' if p >= thresh else 'normal' for p in hate_probs]
    report = classification_report(y_true, preds, labels=['hate', 'offensive', 'normal'], output_dict=True, zero_division=0)
    hate_f1 = report.get('hate', {}).get('f1-score', 0)
    normal_f1 = report.get('normal', {}).get('f1-score', 0)
    macro_f1 = report['macro avg']['f1-score']
    precision = report.get('hate', {}).get('precision', 0)
    recall = report.get('hate', {}).get('recall', 0)
    print(f'{thresh:<12.1f} {hate_f1:<12.4f} {normal_f1:<12.4f} {macro_f1:<12.4f} {precision:<12.4f} {recall:.4f}')

---
## Section 9: Swap to Fine-Tuned SG Model

Once your SG fine-tuned model (Phase 1 of the project) is trained and uploaded to HuggingFace,
swap it in with **zero changes to the guard pipeline**.

This is the core Guardrails value proposition: **model-agnostic validation pipeline**.

In [ ]:
# ── Placeholder: replace MODEL_ID with your published SG model ──────────
# Example:
#   MODEL_ID = 'your-username/singbert-sg-hate-speech'
#   MODEL_ID = 'your-username/roberta-sg-hate-speech'

SG_FINETUNED_MODEL = None  # Set this when your model is ready

if SG_FINETUNED_MODEL:
    # The ONLY change: model_name and hate_label
    # Everything else — guard composition, on_fail actions, benchmark code — stays identical
    sg_finetuned_guard = Guard(name='sg-finetuned-guard').use(
        SinglishHateSpeech,
        model_name=SG_FINETUNED_MODEL,
        hate_threshold=0.6,
        hate_label='hate',  # SG fine-tuned model uses 'hate' not 'hatespeech'
        on_fail='fix',
    )

    # Run the same benchmark
    y_pred_ft = []
    ft_validator = SinglishHateSpeech(model_name=SG_FINETUNED_MODEL, hate_label='hate', on_fail='noop')

    for row in tqdm(sg_test, desc='Fine-tuned guard eval'):
        prob = ft_validator._get_hate_prob(row['text'])
        y_pred_ft.append('hate' if prob >= 0.6 else 'normal')

    print('Fine-tuned SG model guard on test set:')
    print(classification_report(y_true, y_pred_ft, labels=['hate', 'offensive', 'normal'], zero_division=0))
else:
    print('SG_FINETUNED_MODEL not set.')
    print('Set this variable to your HuggingFace model ID once training is complete.')
    print('\nExpected improvement over placeholder:')
    print('  - 3-class output (hate / offensive / normal — not just binary)')
    print('  - Higher hate F1 on Singlish-specific slurs')
    print('  - Lower false positive rate on Singlish expletives')

In [ ]:
# ── Side-by-side comparison template (run after fine-tuning) ──────────
# Uncomment after setting SG_FINETUNED_MODEL above

# comparison_cases = [
#     # Cases where placeholder should fail but SG model should succeed
#     'those ah tiong always come here take our jobs',
#     'all bangla workers should go back to their country',
#     # Cases where placeholder over-triggers but SG model should not
#     'nabeh, you damn stupid lah',
#     'cb, you sabo me again',
# ]
#
# print(f'{"Text":<50} {"Placeholder":<15} {"SG Fine-Tuned"}')
# print('-'*80)
# for text in comparison_cases:
#     ph_result = eval_guard.validate(text)
#     ft_result = sg_finetuned_guard.validate(text)
#     print(f'{text[:48]:<50} {"FAIL" if not ph_result.validation_passed else "pass":<15} {"FAIL" if not ft_result.validation_passed else "pass"}')

---
## Section 10: Publishing to the Guardrails Hub

Once your SG fine-tuned model is validated, you can publish the `SinglishHateSpeech` validator
as a community contribution to the Guardrails Hub — making it available to any developer.

### Steps to publish

```bash
# 1. Use the official validator template
git clone https://github.com/guardrails-ai/validator-template.git singlish-hate-speech
cd singlish-hate-speech

# 2. Replace validator/main.py with your SinglishHateSpeech class
#    (copy from Section 4 of this notebook)

# 3. Update pyproject.toml
#    name = "singlish-hate-speech"
#    description = "Detects hate speech in Singlish text"
#    dependencies = ["transformers", "torch"]

# 4. Write tests in tests/test_validator.py

# 5. Publish
guardrails hub submit .
```

### What the published validator would look like

```bash
# Any developer can then install and use your validator:
guardrails hub install hub://your-username/singlish_hate_speech
```

```python
from guardrails.hub import SinglishHateSpeech
from guardrails import Guard

guard = Guard().use(SinglishHateSpeech, hate_threshold=0.7, on_fail='exception')
guard.validate(user_input)  # Works on any LLM app
```

### Positioning

The Guardrails Hub has 70 validators — **none targeting Singlish or SEA languages**.
This fills a real gap, and aligns with:
- GovTech Singapore's push for Singapore-aware LLM safety rails
- The SGHateCheck benchmark (SUTD) for functional hate speech testing
- Growing enterprise adoption of Guardrails for production LLM safety

Your SG synthetic dataset + fine-tuned model + Guardrails validator forms a complete,
deployable pipeline — not just a research artifact.

In [ ]:
# ── Final summary ────────────────────────────────────────────────────
print("""
=================================================================
SINGLISH GUARDRAILS — SUMMARY
=================================================================

What we built:
  SinglishHateSpeech — a Guardrails AI custom validator that
  wraps any HuggingFace 3-class hate speech classifier.

Current state (placeholder model):
  Model: shareeep/roberta-base-fine-tuned-hatexplain
  Limitation: English-trained, binary output, misses Singlish slurs

After Phase 1 fine-tuning (swap in Section 9):
  Model: SG-specific SingBERT or RoBERTa fine-tuned on 319 SG rows
  Improvement: 3-class, Singlish-aware, trained on 20 SG terms

Deployment options:
  1. Import directly (guard = Guard().use(SinglishHateSpeech, ...))
  2. Guardrails Server (microservice, all apps connect via HTTP)
  3. Publish to Guardrails Hub (community validator)

Next steps:
  [ ] Complete Phase 1 fine-tuning (singbert_sg_experiments.ipynb)
  [ ] Upload best model to HuggingFace Hub
  [ ] Set SG_FINETUNED_MODEL in Section 9 and re-run benchmark
  [ ] Compare vs ToxicLanguage baseline (Section 3)
  [ ] Package and submit to Guardrails Hub
=================================================================
""")